# Proof of Concept of the complete Workflow

The POC should show the complete workflow from the API-call for Text up to the creation of Knowledge Graphs of the extracted (meta)data.
Start with the classes ASB extracted in his CatalysisIE Knowledge Graph and maybe extend it with some additional classes.

Steps:
1) Get a basic description for catalytical reactions from wikipedia introductory section (API).
2) Call LogicMill API to get various paper about this catalytic reaction (API).
3) Call various APIs to get as much abstracts as possible for the papers LogicMill suggested.
4) Hand each abstract to Ollama for the automated (meta)data extraction. (Dump all information in a JSON file. One file for each catalytic reaction)
5) Mapping  
6) Create Knowledge Graph
7) Predefine some queries

# Set list for terms which will be analysed by the workflow
- Hydroformylation
- Methanation
- Haber-Bosch process
- Contact process
- Ostwald process
- Fischer-Tropsch process
- Steam reforming

In [ ]:
# Setup
import re

key = "Steam reforming"
model = "gemma3:4b"

# Contact e-mail used for the "polite pool" of the public APIs (OpenAlex etc.).
# Replace with a real address you control.
CONTACT_EMAIL = "marc.voelkenrath@tu-dortmund.de"

# Filesystem-safe stem for all intermediate files, e.g. "Steam reforming" -> "Steam_reforming".
# Using this everywhere avoids spaces in filenames and keeps the per-cell hand-off consistent.
slug = re.sub(r"[^\w]+", "_", key).strip("_")

In [ ]:
# Call DBpedia for articles (Try later, DBpedia seem to have some problems)
import requests

def get_dbpedia_abstract(resource: str, lang: str = "en") -> str:

    url = f"https://dbpedia.org/data/{resource}.json"
    headers = {"User-Agent": "MetaExtract4Cat/1.0"}

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    data = response.json()

    resource_uri = f"http://dbpedia.org/resource/{resource}"
    abstract_key = "http://dbpedia.org/ontology/abstract"

    abstracts = data.get(resource_uri, {}).get(abstract_key, [])
    
    # Filter by language
    for entry in abstracts:
        if entry.get("lang") == lang:
            return entry.get("value", "")
    
    return ""

text = get_dbpedia_abstract("Catalysis")
print(text)


In [36]:
# Call Wikipedia for article
import requests

def get_wikipedia_plaintext(title: str, lang: str = "en") -> str:
    url = f"https://{lang}.wikipedia.org/api/rest_v1/page/summary/{title}"
    response = requests.get(url, headers={"User-Agent": "MetaExtract4Cat/1.0"})
    response.raise_for_status()
    return response.json().get("extract", "")

wiki_text = get_wikipedia_plaintext(key)
print(wiki_text)

Steam reforming or steam methane reforming (SMR) is a method for producing syngas by reaction of hydrocarbons with water. Commonly, natural gas is the feedstock. The main purpose of this technology is often hydrogen production, although syngas has multiple other uses such as production of ammonia or methanol. The reaction is represented by this equilibrium:


In [ ]:
# Call LogicMill API to get a list of various paper about the catalytic reactions
import json
import os
from pathlib import Path
from urllib3.util import Retry
from requests import Session
from requests.adapters import HTTPAdapter

# API token is loaded from the environment / a repo-root .env file (gitignored).
# Never hard-code secrets in notebooks. See .env.example.
def _load_logicmill_token():
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        for parent in [Path.cwd(), *Path.cwd().parents]:
            env_file = parent / ".env"
            if env_file.exists():
                for line in env_file.read_text(encoding="utf-8").splitlines():
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, _, v = line.partition("=")
                        os.environ.setdefault(k.strip(), v.strip().strip("\"'"))
                break
    token = os.environ.get("LOGICMILL_TOKEN")
    if not token:
        raise RuntimeError(
            "LOGICMILL_TOKEN is not set. Copy .env.example to .env (repo root) and "
            "add your token, then re-run. Never commit the token."
        )
    return token

TOKEN = _load_logicmill_token()

# API settings
URL = "https://api.logic-mill.net/api/v1/graphql/"

QUERY = """
query embedDocumentAndSimilaritySearch($data: [EncodeDocumentPart], $indices: [String], $amount: Int, $model: String!) {
  encodeDocumentAndSimilaritySearch(
    data: $data
    indices: $indices
    amount: $amount
    model: $model
  ) {
    id
    score
    index
    document {
      type
      title
      url
      doi
      publicationDate
      PatspecterEmbedding
    }
  }
}
"""

# Build session once at module level
_session = Session()
_retries = Retry(
    total=5,
    backoff_factor=0.1,
    status_forcelist=[500, 501, 502, 503, 504, 524]
)
_session.mount("https://", HTTPAdapter(max_retries=_retries))
_session.headers.update({
    "content-type": "application/json",
    "Authorization": f"Bearer {TOKEN}"
})

def search_logicmill(title: str, abstract: str, amount: int = 5, model: str = "patspecter") -> list:
    variables = {
        "model": model,
        "data": [
            {"key": "title", "value": title},
            {"key": "abstract", "value": abstract}
        ],
        "amount": amount,
        "indices": ["publications"]
    }

    r = _session.post(URL, json={"query": QUERY, "variables": variables})

    if r.status_code != 200:
        raise ConnectionError(f"LogicMill API error {r.status_code}: {r.text}")

    response = r.json()

    if "errors" in response:
        raise ValueError(f"GraphQL error: {response['errors']}")

    return response["data"]["encodeDocumentAndSimilaritySearch"]

title = key
abstract = wiki_text

results = search_logicmill(title, abstract, amount=25)
print(json.dumps(results, indent=2))

with open(f"response_{slug}.json", "w") as f:
    json.dump(results, f, indent=4)

In [ ]:
# Deduplicate the LogicMill results before enrichment
# LogicMill can return the same publication under different OpenAlex IDs
# (e.g. one record with a DOI and one without). Without this step the same
# paper would be enriched, extracted and counted twice in the knowledge graph.
# Deduplicate by OpenAlex id, normalized DOI and normalized title (exact and
# near-duplicate). Keep the first (highest-scoring) copy and back-fill a DOI
# onto it if it was missing.
import json
import re
from difflib import SequenceMatcher

# Two normalized titles closer than this ratio are treated as the same paper.
_TITLE_SIMILARITY_THRESHOLD = 0.95


def _normalize_id(openalex_id: str) -> str:
    """Strip the host so 'https://openalex.org/W123' and 'W123' compare equal."""
    if not openalex_id:
        return ""
    return openalex_id.rstrip("/").rsplit("/", 1)[-1].strip().upper()


def _normalize_doi(doi: str) -> str:
    if not doi:
        return ""
    d = doi.lower().strip()
    for prefix in ("https://doi.org/", "http://dx.doi.org/", "https://dx.doi.org/", "doi:"):
        if d.startswith(prefix):
            d = d[len(prefix):]
    return d.strip()

def _normalize_title(title: str) -> str:
    if not title:
        return ""
    t = title.lower()
    t = re.sub(r"<[^>]+>", " ", t)      # drop HTML/XML tags, e.g. <sub>
    t = re.sub(r"&[a-z]+;", " ", t)     # drop HTML entities, e.g. &lt;
    t = re.sub(r"[^a-z0-9]+", " ", t)   # punctuation -> space
    return re.sub(r"\s+", " ", t).strip()

def deduplicate(items: list) -> tuple:
    seen_id = {}      # normalized OpenAlex id -> position in `unique`
    seen_doi = {}     # normalized DOI         -> position in `unique`
    seen_title = {}   # normalized title       -> position in `unique`
    unique = []
    duplicates = []

    def _fuzzy_title_match(title_n):
        """Return position of an existing near-duplicate title, else None."""
        if not title_n:
            return None
        for kept_title, pos in seen_title.items():
            if SequenceMatcher(None, title_n, kept_title).ratio() >= _TITLE_SIMILARITY_THRESHOLD:
                return pos
        return None

    for item in items:
        doc = item.get("document", {})
        id_n = _normalize_id(item.get("id", ""))
        doi_n = _normalize_doi(doc.get("doi", ""))
        title_n = _normalize_title(doc.get("title", ""))

        match = None
        if id_n and id_n in seen_id:
            match = seen_id[id_n]
        elif doi_n and doi_n in seen_doi:
            match = seen_doi[doi_n]
        elif title_n and title_n in seen_title:
            match = seen_title[title_n]
        else:
            match = _fuzzy_title_match(title_n)

        if match is not None:
            # Same paper already kept. Back-fill a DOI if the kept copy lacked one.
            kept_doc = unique[match]["document"]
            if not _normalize_doi(kept_doc.get("doi", "")) and doi_n:
                kept_doc["doi"] = doc.get("doi", "")
                seen_doi[doi_n] = match
            duplicates.append(item)
            continue

        pos = len(unique)
        unique.append(item)
        if id_n:
            seen_id[id_n] = pos
        if doi_n:
            seen_doi[doi_n] = pos
        if title_n:
            seen_title[title_n] = pos

    return unique, duplicates

# Load raw LogicMill results
input_file = f"response_{slug}.json"
with open(input_file, "r") as f:
    items = json.load(f)

unique, duplicates = deduplicate(items)

print(f"\n{'='*60}")
print(f"DEDUPLICATION")
print(f"{'='*60}")
print(f"Loaded:             {len(items)}")
print(f"Unique:             {len(unique)}")
print(f"Duplicates removed: {len(duplicates)}")
if duplicates:
    print(f"\nRemoved as duplicates:")
    for d in duplicates:
        doc = d["document"]
        print(f"  - {doc.get('title', '')[:55]}")
        print(f"      id={d.get('id')}, doi={doc.get('doi') or 'N/A'}")

# Save the deduplicated list for the enrichment step
output_file = f"response_{slug}_deduped.json"
with open(output_file, "w") as f:
    json.dump(unique, f, indent=4)

print(f"\nSaved {len(unique)} unique documents to: {output_file}")
print(f"{'='*60}\n")

In [ ]:
# Call OpenAlex, Semantic Scholar, CrossRef API's to get as many abstracts as possible
# Unpaywall was removed: its /v2/ response carries no abstract field, so it could
# never contribute (and only sent a placeholder e-mail).
import html
import json
import re
import requests
from time import sleep
from typing import Optional
from urllib3.util import Retry
from requests.adapters import HTTPAdapter

# Shared session with automatic retry/backoff for the public abstract APIs.
# 429 (rate limit) is the key one here: Semantic Scholar throttles unauthenticated
# traffic, and without this it was silently treated as "no abstract".
_abstract_session = requests.Session()
_abstract_session.headers.update({"User-Agent": f"MetaExtract4Cat/1.0 (mailto:{CONTACT_EMAIL})"})
_abstract_session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=4,
    backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True,
    allowed_methods=frozenset(["GET"]),
)))

# Load the existing response
with open(f"response_{slug}_deduped.json", "r") as f:
    data = json.load(f)


def _clean_text(s: Optional[str]) -> Optional[str]:
    """Unescape HTML entities and strip XML/HTML tags (e.g. JATS <jats:p>, <sub>)
    before the text is stored / handed to the LLM."""
    if not s:
        return s
    s = html.unescape(s)
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", s).strip()


# Trying to get abstract from multiple sources
def get_abstract(openalex_id: str, doi: str, title: str) -> Optional[str]:

    # source 1: OpenAlex via OpenAlex-ID
    abstract = _get_from_openalex(openalex_id)
    if abstract:
        return abstract

    # source 2: semantic scholar with DOI
    if doi and doi.strip():
        abstract = _get_from_semantic_scholar_doi(doi)
        if abstract:
            return abstract

    # source 3: crossref with doi
    if doi and doi.strip():
        abstract = _get_from_crossref(doi)
        if abstract:
            return abstract

    # source 4: semantic scholar with title
    if title and title.strip():
        abstract = _get_from_semantic_scholar_title(title)
        if abstract:
            return abstract

    return None

def _get_from_openalex(openalex_id: str) -> Optional[str]:
    try:
        url = f"https://api.openalex.org/works/{openalex_id}"
        response = _abstract_session.get(url, params={"mailto": CONTACT_EMAIL}, timeout=10)
        if response.status_code == 200:
            work = response.json()

            # try abstract inverted index method
            if "abstract_inverted_index" in work and work["abstract_inverted_index"]:
                try:
                    inverted = work["abstract_inverted_index"]
                    if not inverted:
                        return None
                    max_pos = max(max(positions) for positions in inverted.values())
                    abstract_words = [None] * (max_pos + 1)
                    for word, positions in inverted.items():
                        for pos in positions:
                            abstract_words[pos] = word
                    abstract = " ".join(w for w in abstract_words if w)
                    if abstract:
                        return abstract
                except (ValueError, KeyError, AttributeError) as e:
                    pass

            # try direct abstract field
            if "abstract" in work and work["abstract"]:
                return work["abstract"]
    except requests.Timeout:
        print(f"OpenAlex timeout for {openalex_id}")
    except Exception as e:
        print(f"OpenAlex error for {openalex_id}: {type(e).__name__}")

    return None
            

def _get_from_semantic_scholar_doi(doi: str) -> Optional[str]:
    try:
        url = f"https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}?fields=abstract"
        response = _abstract_session.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if "abstract" in data and data["abstract"]:
                return data["abstract"]
    except requests.Timeout:
        print(f"Semantic Scholar (DOI) timeout for {doi}")
    except Exception as e:
        print(f"Semantic Scholar (DOI) error for {doi}: {type(e).__name__}")
    return None

def _get_from_semantic_scholar_title(title: str) -> Optional[str]:
    try:
        url = "https://api.semanticscholar.org/graph/v1/paper/search"
        params = {"query": title, "fields": "abstract", "limit": 1}
        response = _abstract_session.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get("data") and len(data["data"]) > 0:
                paper = data["data"][0]
                if "abstract" in paper and paper["abstract"]:
                    return paper["abstract"]
    except requests.Timeout:
        print(f"Semantic Scholar (title) timeout for {title}")
    except Exception as e:
        print(f"Semantic Scholar (title) error for {title}: {type(e).__name__}")
    return None
    
def _get_from_crossref(doi: str) -> Optional[str]:
    try:
        url = f"https://api.crossref.org/works/{doi}"
        response = _abstract_session.get(url, params={"mailto": CONTACT_EMAIL}, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get("message", {}).get("abstract"):
                return data["message"]["abstract"]
    except requests.Timeout:
        print(f"Crossref timeout for {doi}")
    except Exception as e:
        print(f"Crossref error for {doi}: {type(e).__name__}")
    return None

# Adding abstracts to document
print(type(data))
items = data #['data']['encodeDocumentAndSimilaritySearch'] --> data is already a list
total = len(items)
success_count = 0
failed_count = 0

print(f"\n{'='*60}")
print(f"Processing {total} documents...")
print(f"{'='*60}\n")

for idx, item in enumerate(items, 1):
    openalex_id = item.get('id', '')
    doi = item['document'].get('doi', '').strip()
    title = item['document'].get('title', '')
    
    print(f"[{idx}/{total}] {title[:50]}...")
    print(f"  ID: {openalex_id}, DOI: {doi if doi else 'N/A'}")
    
    abstract = _clean_text(get_abstract(openalex_id, doi, title))
    
    if abstract:
        item['document']['abstract'] = abstract
        success_count += 1
        print(f"  ✓ Added abstract ({len(abstract)} chars)")
    else:
        item['document']['abstract'] = None
        failed_count += 1
        print(f"  ✗ No abstract found")
    
    sleep(1.0)  # Be nice to the APIs (Semantic Scholar throttles unauthenticated traffic)

# Save updated JSON
output_file = f'response_{slug}_with_abstracts.json'
with open(output_file, 'w') as f:
    json.dump(data, f, indent=2)

print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Total documents: {total}")
print(f"✓ Abstracts added: {success_count} ({success_count/total*100:.1f}%)")
print(f"✗ Abstracts missing: {failed_count} ({failed_count/total*100:.1f}%)")
print(f"\nSaved to: {output_file}")
print(f"{'='*60}\n")

In [ ]:
# Call Ollama (local LLM) for the metadata extraction.
import json
import ollama

MODEL = model

SYSTEM_PROMPT = """
You are a chemistry knowledge extraction assistant.
Extract structured metadata from the given text and return JSON only.

Extract the following:
- reaction (name, reactants, products)
- catalysts (name, type: heterogeneous/homogeneous/enzymatic)
- operating conditions (temperature, pressure)
- synthesis methods
- support materials
- characterization methods

Return strictly valid JSON with the following structure. If a field is not mentioned, set it to null.

Structure:
{{
  "reaction": {{
    "name": "",
    "reactants": [
      ""
    ],
    "products": [
      ""
    ]
  }},
  "catalysts": [
    {{
      "name": "",
      "type": ""
    }}
  ],
  "operating_conditions": {{
    "temperature": "",
    "pressure": ""
  }},
  "synthesis_methods": "",
  "support_materials": "",
  "characterization_methods": ""
}}

Text:
{text}
"""

def extract_metadata(text: str) -> dict:
    response = ollama.generate(
        model=MODEL,
        prompt=SYSTEM_PROMPT.format(text=_clean_text(text)),
        format="json",  # ask Ollama to constrain the output to valid JSON
    )

    content = response["response"]
    # Belt-and-suspenders: strip any code fences in case the model still adds them.
    content = content.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(content)

# Load JSON
input_file = f"response_{slug}_with_abstracts.json"
with open(input_file, "r") as f:
    items = json.load(f)

total = len(items)
success_count = 0
failed_count = 0
error_count = 0

print(f"\n{'='*60}")
print(f"Processing {total} documents...")
print(f"{'='*60}\n")

for idx, item in enumerate(items, 1):
    title = item["document"].get("title", "")
    abstract = item["document"].get("abstract")

    print(f"[{idx}/{total}] {title[:60]}...")

    if not abstract:
        print(f"  — Skipped (no abstract)")
        failed_count += 1
        continue

    try:
        metadata = extract_metadata(abstract)
        item["document"]["extracted_metadata"] = metadata
        success_count += 1
        print(f"  ✓ Metadata extracted")
    except json.JSONDecodeError as e:
        item["document"]["extracted_metadata"] = None
        error_count += 1
        print(f"  ✗ JSON parse error: {e}")
    except Exception as e:
        item["document"]["extracted_metadata"] = None
        error_count += 1
        print(f"  ✗ Extraction error: {type(e).__name__}: {e}")

# Save updated JSON
output_file = f"response_{slug}_with_metadata.json"
with open(output_file, "w") as f:
    json.dump(items, f, indent=2, ensure_ascii=False)

print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Total:     {total}")
print(f"✓ Success: {success_count}")
print(f"— Skipped: {failed_count} (no abstract)")
print(f"✗ Errors:  {error_count}")
print(f"\nSaved to: {output_file}")
print(f"{'='*60}\n")